# 災害救助法適用事例の分析

災害救助法の適用事例のCSVを、地図で落とし込めるように編集

In [1]:
import pandas as pd
import geopandas as gpd
import os
import json

In [2]:
# 災害救助法適用事例をロード
disaster_cases = pd.read_csv(os.path.join('data', '災害救助法適用事例', '災害救助法適用事例.csv'), encoding='shift_jis')

In [3]:
disaster_cases.head()

,年度,西暦年度,決定日,適用日,災害タイトル,災害種別,適用基準,都道府県,市町村
0,H26,2014,2014-07-09,2014-07-09,平成26年台風第８号,台風,4号,長野県,南木曽町
1,H26,2014,2014-07-14,2014-07-09,平成26年台風第８号,台風,1号,山形県,南陽市
2,H26,2014,2014-08-08,2014-08-03,平成26年台風第12号,台風,1号,高知県,いの町
3,H26,2014,2014-08-09,2014-08-09,平成26年台風第11号,台風,4号,高知県,高知市
4,H26,2014,2014-08-09,2014-08-09,平成26年台風第11号,台風,4号,高知県,大豊町


In [4]:
print([x for x in disaster_cases['災害タイトル'].unique()])

['平成26年台風第８号', '平成26年台風第12号', '平成26年台風第11号', '平成26年8月15日からの大雨', '平成26年8月19日からの大雨', '御嶽山噴火による被害', '長野県神城断層地震', '12月5日からの大雪', '口永良部島噴火', '平成27年９月関東・東北豪雨', '平成27年台風第21号', '平成28年熊本地震', '平成28年台風第10号', '平成28年鳥取県中部地震', '平成28年新潟県糸魚川市における大規模火災（住宅火災）', '平成29年7月九州北部豪雨', '平成29年7月22日からの大雨', '平成29年台風第18号', '平成29年台風第21号', '平成30年2月4日からの大雪', '平成29年度豪雪', '平成30年大阪府北部を震源とする地震', '平成30年７月豪雨', '平成30年８月30日からの大雨', '平成30年北海道胆振東部地震', '令和元年８月の前線に伴う大雨', '令和元年台風第15号', '令和元年台風第15号（停電）', '令和元年台風第19号', '令和２年７月３日からの大雨', '令和２年台風第14号', '令和２年12月16日からの大雪（交通障害）', '令和３年１月７日からの大雪', '令和３年福島県沖を震源とする地震', '令和３年栃木県足利市における大規模火災（山林火災）', '令和３年新潟県糸魚川市における地滑り', '島根県松江市における大規模火災（住宅火災）', '令和３年７月１日からの大雨', '台風第９号から変わった温帯低気圧に伴う大雨', '令和３年８月１１日からの大雨', '令和３年長野県茅野市において発生した土石流', '令和４年福島県沖を震源とする地震', '令和４年７月14日からの大雨', '令和４年８月３日からの大雨', '令和４年８月９日からの大雨', '令和４年台風第14号', '令和４年台風第15号', '令和４年12月17日からの大雪（交通障害等）', '令和４年12月22日からの大雪（長期停電）', '令和４年山形県鶴岡市の土砂崩れ', '令和５年１月24日からの大雪（交通障害）', '令和５年石川県能登地方を震源とする地震', '令和５年梅雨前線による大雨及び台風第２号', '令和５年６月29日からの大雨', '令和５年７月７日からの

In [5]:
# 日付を日付形式に変換
disaster_cases['決定日'] = pd.to_datetime(disaster_cases['決定日'])
disaster_cases['適用日'] = pd.to_datetime(disaster_cases['適用日'])

In [6]:
disaster_cases['災害種別'].value_counts()

災害種別
台風      934
大雨      432
地震      149
大雪      146
津波      118
火災        5
噴火        3
地滑り       1
土石流       1
土砂崩れ      1
道路陥没      1
Name: count, dtype: int64

In [7]:
disaster_cases.dtypes

年度                object
西暦年度               int64
決定日       datetime64[ns]
適用日       datetime64[ns]
災害タイトル            object
災害種別              object
適用基準              object
都道府県              object
市町村               object
dtype: object

## 自治体境界データを編集

In [24]:
japan_map = gpd.read_file(os.path.join('data', 'japan_ver85','japan_ver85','japan_ver85.shp'))

In [25]:
japan_map.head()

,JCODE,KEN,SICHO,GUN,SEIREI,SIKUCHOSON,CITY_ENG,P_NUM,H_NUM,Shape_Leng,Shape_Area,geometry
0,01101,北海道,石狩振興局,None,札幌市,中央区,"Sapporo-shi, Chuo-ku",248680,141429,0.542590,0.005128,"POLYGON ((141.34233 43.06682, 141.3552 43.0685..."
1,01102,北海道,石狩振興局,None,札幌市,北区,"Sapporo-shi, Kita-ku",289323,139675,0.567773,0.007031,"POLYGON ((141.40839 43.18395, 141.40427 43.182..."
2,01103,北海道,石狩振興局,None,札幌市,東区,"Sapporo-shi, Higashi-ku",265379,131188,0.397203,0.006289,"POLYGON ((141.44698 43.15513, 141.4486 43.1532..."
3,01104,北海道,石狩振興局,None,札幌市,白石区,"Sapporo-shi, Shiroishi-ku",211835,108233,0.308211,0.003815,"POLYGON ((141.46569 43.1, 141.46812 43.09704, ..."
4,01105,北海道,石狩振興局,None,札幌市,豊平区,"Sapporo-shi, Toyohira-ku",225298,118650,0.428316,0.005101,"POLYGON ((141.38479 43.0466, 141.38558 43.0472..."


In [26]:
# 政令指定都市が区ごとに別れているため、市全体を一つのポリゴンとしてまとめる
# SEIREI値を持つ行と持たない行を分離する
japan_map_with_seirei = japan_map[japan_map['SEIREI'].notna()].copy()
japan_map_without_seirei = japan_map[japan_map['SEIREI'].isna()].copy()

# 政令市フラグ
japan_map_with_seirei['seirei_flag'] = 1
japan_map_without_seirei['seirei_flag'] = 0

# 政令市の情報を市区町村列にコピー
japan_map_with_seirei['SIKUCHOSON'] = japan_map_with_seirei['SEIREI'] + japan_map_with_seirei['SIKUCHOSON']

# 政令市の一覧を取得
seirei_list = japan_map_with_seirei[['KEN','SEIREI']].drop_duplicates().agg(' '.join, axis=1).to_list()

# # 同じSEIREI値を持つポリゴンを統合する
# japan_map_merged_with_seirei = japan_map_with_seirei.dissolve(by='SEIREI', aggfunc='first')

# # インデックスをリセットしてSEIREIを列として再度作成する
# japan_map_merged_with_seirei = japan_map_merged_with_seirei.reset_index()

# # SEIREIの値をSIKUCHOSONにコピーする
# japan_map_merged_with_seirei['SIKUCHOSON'] = japan_map_merged_with_seirei['SEIREI']

# # CITY_ENG列で、カンマ以降のすべてを省略する（区が入っている部分を消去）
# japan_map_merged_with_seirei['CITY_ENG'] = japan_map_merged_with_seirei['CITY_ENG'].str.split(',').str[0]

# # JCODEの最後の文字を0に置き換える（区の自治体コードとなっているのを市の自治体コードに変換する）
# japan_map_merged_with_seirei['JCODE'] = japan_map_merged_with_seirei['JCODE'].str[:-1] + '0'

# 政令市部分とそれ以外を統合し、SEIREI列を削除する
# japan_map = pd.concat([japan_map_merged_with_seirei, japan_map_without_seirei], ignore_index=True)

japan_map = pd.concat([japan_map_with_seirei, japan_map_without_seirei], ignore_index=True)
japan_map = japan_map.drop(columns=['SEIREI'])

# JCODE列でソートする
japan_map.sort_values('JCODE', inplace=True)

# 不要な列を削除
japan_map.drop(columns=['P_NUM', 'H_NUM', 'Shape_Area', 'Shape_Leng'], inplace=True)

In [27]:
japan_map.head()

,JCODE,KEN,SICHO,GUN,SIKUCHOSON,CITY_ENG,geometry,seirei_flag
0,01101,北海道,石狩振興局,None,札幌市中央区,"Sapporo-shi, Chuo-ku","POLYGON ((141.34233 43.06682, 141.3552 43.0685...",1
1,01102,北海道,石狩振興局,None,札幌市北区,"Sapporo-shi, Kita-ku","POLYGON ((141.40839 43.18395, 141.40427 43.182...",1
2,01103,北海道,石狩振興局,None,札幌市東区,"Sapporo-shi, Higashi-ku","POLYGON ((141.44698 43.15513, 141.4486 43.1532...",1
3,01104,北海道,石狩振興局,None,札幌市白石区,"Sapporo-shi, Shiroishi-ku","POLYGON ((141.46569 43.1, 141.46812 43.09704, ...",1
4,01105,北海道,石狩振興局,None,札幌市豊平区,"Sapporo-shi, Toyohira-ku","POLYGON ((141.38479 43.0466, 141.38558 43.0472...",1


## データの結合

データの結合ができる状態になっていることを確認したうえで、市区町村ごとに災害データを整理

In [28]:
# 表記揺れを修正
japan_map['SIKUCHOSON'] = japan_map['SIKUCHOSON'].replace({
    '須惠町': '須恵町',
})
disaster_cases['市町村'] = disaster_cases['市町村'].replace({
    '塩竃市': '塩竈市',
    '篠山市': '丹波篠山市', # 2020年に篠山市は丹波篠山市に改称
    '飛?市': '飛騨市',
    '桧枝岐村': '檜枝岐村',
    '梼原町': '檮原町',
})

In [29]:
# 自治体名でマージするためのキーを作成
disaster_cases['prefcity'] = disaster_cases[['都道府県','市町村']].agg(' '.join, axis=1)
japan_map['prefcity'] = japan_map[['KEN','SIKUCHOSON']].agg(' '.join, axis=1)


In [30]:
# 自治体名でマージできることを確認
merged = pd.merge(disaster_cases, japan_map[['prefcity', 'geometry']], on='prefcity', how='left', indicator=True)

In [31]:
merged[merged['_merge'] == 'left_only']['prefcity'].unique()

array(['広島県 広島市', '宮城県 仙台市', '熊本県 全市町村', '大阪府 大阪市', '岡山県 岡山市', '北海道 全市町村',
       '神奈川県 川崎市', '神奈川県 相模原市', '埼玉県 さいたま市', '福岡県 北九州市', '福岡県 福岡市',
       '熊本県 熊本市', '静岡県 静岡市', '静岡県 浜松市', '新潟県 新潟市'], dtype=object)

In [85]:
# 災害データベースを作成
disaster_database = disaster_cases.groupby(['西暦年度','災害タイトル','災害種別'])['適用日'].min().reset_index()
disaster_database['適用日'] = disaster_database['適用日'].dt.strftime('%Y-%m-%d')  # 日付を文字列に変換
disaster_database_dict = disaster_database.to_dict('index')
disaster_database['ID'] = disaster_database.index

In [86]:
# IDを付与
disaster_cases_iter = disaster_cases.merge(disaster_database.drop('適用日', axis=1), on=['西暦年度','災害タイトル','災害種別'], how='left')

In [87]:
disaster_cases_iter.head()

,年度,西暦年度,決定日,適用日,災害タイトル,災害種別,適用基準,都道府県,市町村,prefcity,ID
0,H26,2014,2014-07-09,2014-07-09,平成26年台風第８号,台風,4号,長野県,南木曽町,長野県 南木曽町,5
1,H26,2014,2014-07-14,2014-07-09,平成26年台風第８号,台風,1号,山形県,南陽市,山形県 南陽市,5
2,H26,2014,2014-08-08,2014-08-03,平成26年台風第12号,台風,1号,高知県,いの町,高知県 いの町,4
3,H26,2014,2014-08-09,2014-08-09,平成26年台風第11号,台風,4号,高知県,高知市,高知県 高知市,3
4,H26,2014,2014-08-09,2014-08-09,平成26年台風第11号,台風,4号,高知県,大豊町,高知県 大豊町,3


In [88]:
japan_map.head()

,JCODE,KEN,SICHO,GUN,SIKUCHOSON,CITY_ENG,geometry,seirei_flag,prefcity
0,01101,北海道,石狩振興局,None,札幌市中央区,"Sapporo-shi, Chuo-ku","POLYGON ((141.34233 43.06682, 141.3552 43.0685...",1,北海道 札幌市中央区
1,01102,北海道,石狩振興局,None,札幌市北区,"Sapporo-shi, Kita-ku","POLYGON ((141.40839 43.18395, 141.40427 43.182...",1,北海道 札幌市北区
2,01103,北海道,石狩振興局,None,札幌市東区,"Sapporo-shi, Higashi-ku","POLYGON ((141.44698 43.15513, 141.4486 43.1532...",1,北海道 札幌市東区
3,01104,北海道,石狩振興局,None,札幌市白石区,"Sapporo-shi, Shiroishi-ku","POLYGON ((141.46569 43.1, 141.46812 43.09704, ...",1,北海道 札幌市白石区
4,01105,北海道,石狩振興局,None,札幌市豊平区,"Sapporo-shi, Toyohira-ku","POLYGON ((141.38479 43.0466, 141.38558 43.0472...",1,北海道 札幌市豊平区


In [89]:
japan_map['prefcity']

0       北海道 札幌市中央区
1        北海道 札幌市北区
2        北海道 札幌市東区
3       北海道 札幌市白石区
4       北海道 札幌市豊平区
           ...    
804      千葉県 所属不明地
867      東京都 所属不明地
1210     愛知県 所属不明地
1864    鹿児島県 所属不明地
1906     沖縄県 所属不明地
Name: prefcity, Length: 1907, dtype: object

In [90]:
japan_map['JCODE'].dropna().unique()

array(['01101', '01102', '01103', ..., '47375', '47381', '47382'],
      dtype=object)

In [91]:
# 市町村ごとの災害一覧と集計をdictで作成
disaster_summary = {}
disaster_types = disaster_cases['災害種別'].unique()

for jcode in japan_map['JCODE'].dropna().unique():
    disaster_summary[jcode] = {
        'name': japan_map[japan_map['JCODE'] == jcode]['prefcity'].values[0],
        'disaster_id': [],
        'disaster_count': {disaster_type: 0 for disaster_type in disaster_types}
    }

# 災害データベースに発生自治体の一覧を追加
for item in disaster_database_dict.values():
    item['municipalities'] = []

# 履歴を1行ずつ処理していく
for i, row in disaster_cases_iter.iterrows():
    prefcity = row['prefcity']
    disaster_id = row['ID']
    disaster_type = row['災害種別']
    
    if '全市町村' in prefcity:
        # 県内全域に適用される災害の場合、県内のすべての市町村に災害IDを追加し、災害種別のカウントを増やす
        prefecture = row['都道府県']
        for jcode, summary in disaster_summary.items():
            if summary['name'].startswith(prefecture):
                disaster_summary[jcode]['disaster_id'].append(disaster_id)
                disaster_summary[jcode]['disaster_count'][disaster_type] += 1
                disaster_database_dict[disaster_id]['municipalities'].append(jcode) 
    elif prefcity in seirei_list:
        # 政令市の場合、すべての区に災害IDを追加し、災害種別のカウントを増やす
        for jcode, summary in disaster_summary.items():
            if summary['name'].startswith(prefcity):
                disaster_summary[jcode]['disaster_id'].append(disaster_id)
                disaster_summary[jcode]['disaster_count'][disaster_type] += 1
                disaster_database_dict[disaster_id]['municipalities'].append(jcode)
    else:
        # 通常の市町村の場合、災害IDを追加し、災害種別のカウントを増やす
        for jcode, summary in disaster_summary.items():
            if summary['name'] == prefcity:
                disaster_summary[jcode]['disaster_id'].append(disaster_id)
                disaster_summary[jcode]['disaster_count'][disaster_type] += 1
                disaster_database_dict[disaster_id]['municipalities'].append(jcode)

# 災害の合計値
for jcode, summary in disaster_summary.items():
    summary['disaster_count']['合計'] = sum(summary['disaster_count'].values())


In [92]:
disaster_database_dict

{0: {'西暦年度': 2014,
  '災害タイトル': '12月5日からの大雪',
  '災害種別': '大雪',
  '適用日': '2014-12-08',
  'municipalities': ['36208', '36468', '36489']},
 1: {'西暦年度': 2014,
  '災害タイトル': '平成26年8月15日からの大雨',
  '災害種別': '大雨',
  '適用日': '2014-08-17',
  'municipalities': ['26201', '28223']},
 2: {'西暦年度': 2014,
  '災害タイトル': '平成26年8月19日からの大雨',
  '災害種別': '大雨',
  '適用日': '2014-08-20',
  'municipalities': ['34101',
   '34102',
   '34103',
   '34104',
   '34105',
   '34106',
   '34107',
   '34108']},
 3: {'西暦年度': 2014,
  '災害タイトル': '平成26年台風第11号',
  '災害種別': '台風',
  '適用日': '2014-08-09',
  'municipalities': ['39201', '39344', '39412', '36368']},
 4: {'西暦年度': 2014,
  '災害タイトル': '平成26年台風第12号',
  '災害種別': '台風',
  '適用日': '2014-08-03',
  'municipalities': ['39386']},
 5: {'西暦年度': 2014,
  '災害タイトル': '平成26年台風第８号',
  '災害種別': '台風',
  '適用日': '2014-07-09',
  'municipalities': ['20423', '06213']},
 6: {'西暦年度': 2014,
  '災害タイトル': '御嶽山噴火による被害',
  '災害種別': '噴火',
  '適用日': '2014-09-27',
  'municipalities': ['20432', '20429']},
 7: {'西暦年度': 2014,
 

In [93]:
disaster_summary

{'01101': {'name': '北海道 札幌市中央区',
  'disaster_id': [21],
  'disaster_count': {'台風': 0,
   '大雨': 0,
   '噴火': 0,
   '地震': 1,
   '大雪': 0,
   '火災': 0,
   '地滑り': 0,
   '土石流': 0,
   '土砂崩れ': 0,
   '道路陥没': 0,
   '津波': 0,
   '合計': 1}},
 '01102': {'name': '北海道 札幌市北区',
  'disaster_id': [21],
  'disaster_count': {'台風': 0,
   '大雨': 0,
   '噴火': 0,
   '地震': 1,
   '大雪': 0,
   '火災': 0,
   '地滑り': 0,
   '土石流': 0,
   '土砂崩れ': 0,
   '道路陥没': 0,
   '津波': 0,
   '合計': 1}},
 '01103': {'name': '北海道 札幌市東区',
  'disaster_id': [21],
  'disaster_count': {'台風': 0,
   '大雨': 0,
   '噴火': 0,
   '地震': 1,
   '大雪': 0,
   '火災': 0,
   '地滑り': 0,
   '土石流': 0,
   '土砂崩れ': 0,
   '道路陥没': 0,
   '津波': 0,
   '合計': 1}},
 '01104': {'name': '北海道 札幌市白石区',
  'disaster_id': [21],
  'disaster_count': {'台風': 0,
   '大雨': 0,
   '噴火': 0,
   '地震': 1,
   '大雪': 0,
   '火災': 0,
   '地滑り': 0,
   '土石流': 0,
   '土砂崩れ': 0,
   '道路陥没': 0,
   '津波': 0,
   '合計': 1}},
 '01105': {'name': '北海道 札幌市豊平区',
  'disaster_id': [21],
  'disaster_count': {'台風': 0,
   '大雨': 0,


In [94]:
# JSONとして保存

# 保存するためのフォルダを作成
os.makedirs('disaster_data', exist_ok=True)

with open(os.path.join('disaster_data', 'disaster_database.json'), 'w', encoding='utf-8') as f:
    json.dump(disaster_database_dict, f, ensure_ascii=False, indent=4)
with open(os.path.join('disaster_data', 'disaster_summary.json'), 'w', encoding='utf-8') as f:
    json.dump(disaster_summary, f, ensure_ascii=False, indent=4)

japan_map.to_file(os.path.join('disaster_data', 'japan_map.geojson'), driver='GeoJSON', encoding='utf-8')